In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("./weather.csv")
df.head()

,تاریخ و زمان,نوع سنسور,مقدار,واحد
0,2026-05-11 16:59:03,رطوبت,30.2,%
1,2026-05-11 16:59:03,دما,26.4,°C
2,2026-05-11 16:59:03,فشار هوا,91.5,kPa
3,2026-05-11 16:59:03,تابش خورشیدی,163,W/m2
4,2026-05-11 16:59:03,سرعت باد,16.6,km/h


In [3]:
sensors = df["نوع سنسور"].unique()
sensor_values = dict()
for sensor in sensors:
    sensor_values[sensor] = df[df["نوع سنسور"] == sensor].drop("نوع سنسور", axis=1)
sensor_values[sensor]

,تاریخ و زمان,مقدار,واحد
6,2026-05-11 16:59:03,88.3,degree
7,2026-05-11 16:49:03,93.4,degree
20,2026-05-11 16:39:02,70.2,degree
26,2026-05-11 16:19:00,52.7,degree
32,2026-05-11 16:08:59,73.9,degree
...,...,...,...
46674,2026-03-21 04:16:31,234.8,degree
46679,2026-03-21 04:06:30,143.3,degree
46687,2026-03-21 03:56:29,174.6,degree
46695,2026-03-21 03:46:28,197.3,degree


In [4]:
import pandas as pd
import numpy as np
from datetime import datetime

# ============================================================================
# READ THE LONG-FORMAT DATA
# ============================================================================

df = pd.read_csv('./weather.csv')

# Expected input columns: 'تاریخ و زمان', 'نوع سنسور', 'مقدار', 'واحد'
# Adjust column names based on your actual CSV
if len(df.columns) == 4:
    df.columns = ['datetime_str', 'sensor_type', 'value', 'unit']

# ============================================================================
# PARSE DATETIME
# ============================================================================

df['datetime'] = pd.to_datetime(df['datetime_str'])

# Extract time (HH:MM)
df['time'] = df['datetime'].dt.strftime('%H:%M')

# Try to convert to Persian date
try:
    import jdatetime
    df['persian_date'] = df['datetime'].apply(
        lambda x: jdatetime.date.fromgregorian(date=x).strftime('%Y/%m/%d')
    )
except ImportError:
    # Fallback: 2026 ≈ 1405
    df['persian_date'] = '1405/' + df['datetime'].dt.strftime('%m/%d')

# ============================================================================
# CONVERT VALUE TO NUMERIC (REMOVE ANY UNITS)
# ============================================================================

df['value'] = pd.to_numeric(df['value'], errors='coerce')

# ============================================================================
# MAP SENSOR NAMES
# ============================================================================

sensor_map = {
    'رطوبت': 'humidity_pct',
    'دما': 'temp_C',
    'فشار هوا': 'pressure_kPa',
    'تابش خورشیدی': 'solar_Wm2',
    'سرعت باد': 'wind_kmh',
    'باران': 'rain_mm',
    'جهت باد': 'wind_dir',
}

df['sensor'] = df['sensor_type'].map(sensor_map)

# Remove rows where sensor type couldn't be mapped
df = df.dropna(subset=['sensor'])

# ============================================================================
# ROUND TIME TO NEAREST 10-MINUTE INTERVAL
# ============================================================================

def round_to_10min(dt):
    """Round datetime to nearest 10 minutes"""
    minutes = dt.minute
    rounded_minutes = round(minutes / 10) * 10
    if rounded_minutes == 60:
        return dt.replace(minute=0, second=0, microsecond=0) + pd.Timedelta(hours=1)
    return dt.replace(minute=rounded_minutes, second=0, microsecond=0)

df['datetime_rounded'] = df['datetime'].apply(round_to_10min)
df['time_rounded'] = df['datetime_rounded'].dt.strftime('%H:%M')

# Also round the Persian date if crossing midnight
df['persian_date_rounded'] = df['datetime_rounded'].apply(
    lambda x: jdatetime.date.fromgregorian(date=x).strftime('%Y/%m/%d')
    if 'jdatetime' in dir() else '1405/' + x.strftime('%m/%d')
)

# ============================================================================
# PIVOT FROM LONG TO WIDE FORMAT (NUMERIC VALUES)
# ============================================================================

# First aggregate by datetime and sensor (if multiple readings per 10-min)
df_agg = df.groupby(['datetime_rounded', 'time_rounded', 'persian_date_rounded', 'sensor'])['value'].mean().reset_index()

# Now pivot to wide format
df_wide = df_agg.pivot_table(
    index=['datetime_rounded', 'time_rounded', 'persian_date_rounded'],
    columns='sensor',
    values='value',
    aggfunc='mean'  # Now works because values are numeric!
).reset_index()

# ============================================================================
# CREATE FORMATTED OUTPUT (WITH UNITS)
# ============================================================================

# Create the 'زمان' column
df_wide['زمان'] = df_wide['time_rounded'] + '                                                     ' + df_wide['persian_date_rounded']

# Build output dataframe with formatted values
output = pd.DataFrame()
output['زمان'] = df_wide['زمان']

# Add each column with units
column_formats = {
    'rain_mm': ('باران 6 (mm)', '{:.1f}mm'),
    'solar_Wm2': ('تابش خورشیدی 7 (W/m2)', '{:.0f}W/m2'),
    'wind_dir': ('جهت باد 5 (degree)', '{:.1f}degree'),
    'temp_C': ('دما 1 (°C)', '{:.1f}°C'),
    'humidity_pct': ('رطوبت 2 (%)', '{:.1f}%'),
    'wind_kmh': ('سرعت باد 4 (km/h)', '{:.1f}km/h'),
    'pressure_kPa': ('فشار هوا 3 (kPa)', '{:.1f}kPa'),
}

for eng_name, (persian_name, fmt) in column_formats.items():
    if eng_name in df_wide.columns:
        output[persian_name] = df_wide[eng_name].apply(
            lambda x: fmt.format(x) if pd.notna(x) else '--'
        )
    else:
        output[persian_name] = '--'

# ============================================================================
# SORT BY TIME
# ============================================================================

output = output.sort_values('زمان').reset_index(drop=True)

# ============================================================================
# SAVE TO CSV
# ============================================================================

output.to_csv('weather_conditions_formatted.csv', index=False, encoding='utf-8')

# ============================================================================
# PRINT SUMMARY
# ============================================================================

print(f"Original records: {len(df)}")
print(f"After 10-min aggregation: {len(df_wide)}")
print(f"Date range: {df_wide['time_rounded'].min()} to {df_wide['time_rounded'].max()}")
print(f"Persian dates: {df_wide['persian_date_rounded'].min()} to {df_wide['persian_date_rounded'].max()}")
print(f"\nFirst 5 rows of output:")
print(output.head())
print(f"\n✓ Saved to 'weather_conditions_formatted.csv'")

Original records: 46704
After 10-min aggregation: 6344
Date range: 00:00 to 23:50
Persian dates: 1405/03/21 to 1405/05/11

First 5 rows of output:
                                                زمان باران 6 (mm)  \
0  00:00                                         ...      290.2mm   
1  00:00                                         ...      290.6mm   
2  00:00                                         ...      290.6mm   
3  00:00                                         ...      290.6mm   
4  00:00                                         ...      295.0mm   

  تابش خورشیدی 7 (W/m2) جهت باد 5 (degree) دما 1 (°C) رطوبت 2 (%)  \
0                 0W/m2         39.3degree     12.0°C       60.6%   
1                 0W/m2        303.2degree     11.1°C       68.7%   
2                 0W/m2        343.8degree     15.5°C       53.7%   
3                 0W/m2        305.2degree     18.4°C       47.7%   
4                 0W/m2        165.2degree     11.8°C       87.4%   

  سرعت باد 4 (km/h) فشا